# GSE132080 processing decisions — t_79ff033e

This notebook is the executable reconstruction layer for the source-exhaustive OBS/VAR curation of `prism_collection/GSE132080`. It distinguishes accepted source facts, current accepted Lamin state, proposed append-only revisions, and pending execution evidence.

## Accepted source facts

- NCBI GEO `GSE132080` publishes five supplementary files bound in `source_manifest.json` by URL, byte size, Last-Modified, and SHA-256.
- The raw UMI matrix is 33,694 genes × 23,633 barcodes. The published cell-identities table has 23,608 unique cell barcodes; the exact 25-barcode set difference has no identity row and is excluded, not silently dropped.
- The 33,694 source Ensembl IDs are unique; 34 source symbols are duplicated (68 rows). The immutable X axis retains those source symbols, so VAR must preserve row count/order while exposing the unique Ensembl axis explicitly.
- The 128 experimental sgRNAs map exactly from cell `guide_identity` after removing only its redundant leading target token. Published non-targeting controls and `*` unassigned cells have no sequence row and remain `unknown`, never invented.

In [ ]:
import json
from pathlib import Path

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = next(parent for parent in Path.cwd().parents if (parent / 'pyproject.toml').exists())
evidence = root / 'artifacts/schema_audit/real_dataset_curation_20260722/geo_GSE132080/t_79ff033e'
manifest = json.loads((evidence / 'source_manifest.json').read_text())
inspection = json.loads((evidence / 'inspection_report.json').read_text())
plan = json.loads((evidence / 'plan_receipt.json').read_text())
assert len(manifest['files']) == 5
assert manifest['denominator_accounting']['excluded_unassigned_barcodes'] == 25
assert inspection['invariants']['writes'] == 0
assert plan['status'] == 'PASS' and plan['mode'] == 'plan'
assert plan['registry_counts']['before'] == plan['registry_counts']['after']
plan['member_before']['source_join'], plan['member_before']['var_verdict']

## Current accepted state and planned append-only changes

Current accepted identities are OBS `lhR6Ny3n8QcVeItH0002`, X `NEbod0p6ws0H5wug0000`, and VAR `GJ1HqkBSHfDD1o4m0001`. Read-only planning proved a zero-mismatch exact source→OBS join, immutable X shape/axis binding, and zero registry writes.

The planned OBS revision materializes canonical source-backed metadata, explicit per-field `known`/`unknown`/`not_applicable` state, GSM mapping by gemgroup, partial guide sequences, and source guide phenotypes. It preserves all raw columns, `original_obs_index`, `obs_uuid`, row order, and OBS→X linkage.

The planned VAR revision preserves all 33,694 rows, source-symbol index (including duplicate symbols), and exact X-axis order. It adds `stable_feature_id_namespace`, `organism`, and an explicit unique `feature_index` equal to the exact GEO Ensembl axis. X remains immutable; only the X→VAR feature link advances.

## Pending execution gate

Production mutation is intentionally not run from this notebook. The task requires a bounded writer lease no longer than six hours, but the current approved launcher rejects `--lease-minutes` and `--absolute-max-minutes` unless `--verify-only` is selected, while its writer path enforces a minimum eight-hour lease. The exact packet command therefore fails admission. Mutation, replay/no-op verification, zero-write verification, and final strict audit remain pending until the launcher/card contract is reconciled.